In [ ]:
#| default_exp env

In [ ]:
#| export
from __future__ import annotations
import asyncio, inspect, os, sys, threading
from fastcore.all import Path

In [ ]:
#| export
BUNDLE_ONLY = ('PYTHONHOME', 'PYTHONPATH', 'PYTHONEXECUTABLE', '__PYVENV_LAUNCHER__', 'RESOURCEPATH')

class EnvError(RuntimeError): pass

In [ ]:
#| export
def strip_bundle(env, frozen=None):
    """`env` without a frozen host's interpreter redirection, unchanged where there is none.

    py2app and py2exe point `PYTHONHOME` and `PYTHONPATH` at the bundle so its own helper starts.
    A child that keeps them imports the bundle's standard library under another interpreter, which
    fails somewhere unrelated: `nbdev-test` run from a bundled app died inside `linecache` before
    reporting a single notebook. Outside a bundle this returns what it was given, untouched.
    """
    if not (getattr(sys, 'frozen', False) if frozen is None else frozen): return env
    for name in BUNDLE_ONLY: env.pop(name, None)
    env['PYTHONUTF8'] = '1'
    return env

def clean_env():
    "This process's environment, safe to hand to a child."
    return strip_bundle(os.environ.copy(), frozen=True)

In [ ]:
#| export
def venv_env(python=None, env=None):
    """`env` (this process's, by default) with `python`'s virtual environment in front of it.

    `python` is a path to an interpreter, which is how a caller says "run this project's commands
    in this project's environment". Passing None claims nothing about the environment beyond the
    bundle hygiene above.
    """
    env = strip_bundle(dict(os.environ if env is None else env))
    if not python: return env
    bindir = str(Path(python).parent)
    env['VIRTUAL_ENV'] = str(Path(bindir).parent)
    # Not `UV_PROJECT_ENVIRONMENT`. It is read wherever the process ends up rather than where it
    # started, so a `uv sync` run in another checkout syncs that project's lock into this venv and
    # prunes everything the lock does not name. uv finds the right environment from the directory.
    env.pop('UV_PROJECT_ENVIRONMENT', None)
    env['PATH'] = bindir + os.pathsep + env.get('PATH', '')
    env.pop('PYTHONHOME', None)
    return env

In [ ]:
#| export
def _dockeasy():
    try: from dockeasy.core import env_get, env_set, secret_get, secret_set
    except ImportError as e:
        raise EnvError('environment values are stored by dockeasy: pip install "pullup[store]"') from e
    import logging
    logging.getLogger('dotenv.main').setLevel(logging.ERROR)
    return env_get, env_set, secret_get, secret_set

def _call(fn, *args, **kwargs):
    "Call `fn`, awaiting it on its own loop when dockeasy hands back a coroutine."
    value = fn(*args, **kwargs)
    if not inspect.isawaitable(value): return value
    try: asyncio.get_running_loop()
    except RuntimeError: return asyncio.run(value)
    out = []
    def run():
        try: out.append((True, asyncio.run(value)))
        except BaseException as e: out.append((False, e))
    thread = threading.Thread(target=run); thread.start(); thread.join()
    ok, value = out[0]
    if ok: return value
    raise value

In [ ]:
#| export
class EnvStore:
    "The environment keys one project cares about, read and written through dockeasy."
    def __init__(self, service='fastops', path=None): self.service, self.path = service, path
    def get(self, key, secret=True):
        "One value, from the keychain or the env file, falling back to this process's environment."
        env_get, _set, secret_get, _sset = _dockeasy()
        stored = (_call(secret_get, key, service=self.service, path=self.path) if secret
                  else _call(env_get, key, path=self.path))
        return stored or os.environ.get(key) or ''
    def set(self, key, value, secret=True):
        "Store one value. A secret goes to the keychain as well as the file; a variable does not."
        key = str(key or '').strip()
        if not key: raise EnvError('a key is required')
        if not str(value): raise EnvError(f'{key} needs a value')
        env_get, env_set, secret_get, secret_set = _dockeasy()
        if secret: _call(secret_set, key, str(value), service=self.service, path=self.path)
        else: _call(env_set, key, str(value), path=self.path)
        return {'key': key, 'secret': bool(secret)}
    def unset(self, key):
        "Remove a value from every place `set` put it. Says which places actually held one."
        from dockeasy.core import _FASTOPS_ENV
        from dotenv import unset_key
        key, gone = str(key), []
        try:
            import keyring
            if keyring.get_password(self.service, key) is not None:
                keyring.delete_password(self.service, key)
                gone.append('keychain')
        except Exception: pass
        path = str(self.path or _FASTOPS_ENV)
        try:
            removed, _ = unset_key(path, key)
            if removed: gone.append('env file')
        except Exception: pass
        os.environ.pop(key, None)
        return {'key': key, 'removed': gone}
    def values(self, keys, secret=True):
        "Every key that has a value, as a dict. A key with none is absent rather than empty."
        out = {}
        for k in keys:
            try: v = self.get(k, secret=secret)
            except Exception: v = os.environ.get(k) or ''
            if v: out[k] = v
        return out

In [ ]:
#| export
def env_value(store, key, default='', secret=False):
    "One value out of an environment store, or `default` when it is unset or unreadable."
    try: return store.get(key, secret=secret) or default
    except Exception: return default